# Experiment: METR-LA Exploration

This notebook is our starting point for understanding the METR-LA dataset before we choose a forecasting target, preprocessing strategy, and baseline model.

Questions to answer:
- What is the shape and time coverage of the dataset?
- How many sensors are present and how complete is the data?
- What does traffic speed look like for a few representative sensors?
- What forecasting setup should we use first?


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)


## Paths and dependency check

The METR-LA speeds live in an HDF5 file. Pandas needs the optional `tables` package to read it. This cell makes that requirement explicit so setup problems are easy to diagnose.


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() or (candidate / "data").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
data_dir = repo_root / "data" / "raw" / "METR-LA-Complete"
h5_path = data_dir / "metr-la.h5"
distances_path = data_dir / "distances_la_2012.csv"
sensor_locations_path = data_dir / "graph_sensor_locations.csv"

try:
    import tables  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Missing optional dependency 'tables'. Install it in the project environment before reading metr-la.h5."
    ) from exc

print(f"Repo root: {repo_root}")
print(f"HDF5 file exists: {h5_path.exists()} -> {h5_path}")


## Load the core traffic matrix

Rows are timestamps and columns are sensors. Values are traffic speeds. We also infer the sampling interval so we can decide on forecast horizons later.


In [ ]:
def load_metr_la_hdf(path: Path) -> pd.DataFrame:
    with tables.open_file(path, mode="r") as h5_file:
        node = h5_file.root.df
        values = node.block0_values.read()
        columns = [
            col.decode() if isinstance(col, (bytes, bytearray)) else str(col)
            for col in node.axis0.read()
        ]
        index = pd.to_datetime(node.axis1.read())
    return pd.DataFrame(values, index=index, columns=columns).sort_index()


df = load_metr_la_hdf(h5_path)

time_deltas = df.index.to_series().diff().dropna()
sampling_interval = time_deltas.mode().iloc[0] if not time_deltas.empty else pd.NaT

print(f"Shape: {df.shape}")
print(f"Time range: {df.index.min()} to {df.index.max()}")
print(f"Most common sampling interval: {sampling_interval}")
df.head()


## Dataset profile

This gives us a quick summary of missing values, spread, and the most incomplete sensors.


In [ ]:
sensor_missing = df.isna().sum().sort_values(ascending=False)
profile = pd.Series(
    {
        "num_timestamps": df.shape[0],
        "num_sensors": df.shape[1],
        "total_missing_values": int(df.isna().sum().sum()),
        "missing_fraction": float(df.isna().mean().mean()),
        "global_mean_speed": float(df.stack().mean()),
        "global_std_speed": float(df.stack().std()),
        "global_min_speed": float(df.stack().min()),
        "global_max_speed": float(df.stack().max()),
    }
)

display(profile.to_frame("value"))
display(sensor_missing.head(10).to_frame("missing_values"))


## Summary statistics by sensor

Looking across sensors helps us spot outliers or strange ranges before we train anything.


In [ ]:
sensor_stats = df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
sensor_stats.sort_values("mean").head(10)


## Plot a few sensor time series

A small random sample is enough to verify seasonality, spikes, and missing stretches.


In [ ]:
sample_sensor_count = min(4, df.shape[1])
sample_sensors = list(df.columns[:sample_sensor_count])

ax = df[sample_sensors].iloc[: 24 * 12 * 7].plot(figsize=(14, 6), alpha=0.8)
ax.set_title("Sample METR-LA sensor speeds (first 7 days)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Speed")
plt.legend(title="Sensor", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()


## Aggregate daily pattern

This view helps us see whether there is a strong repeating daily traffic rhythm, which is useful for both baselines and feature engineering.


In [ ]:
average_by_time = df.groupby(df.index.time).mean().mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(average_by_time)), average_by_time.values, linewidth=2)
tick_positions = np.linspace(0, len(average_by_time) - 1, 8, dtype=int)
ax.set_xticks(tick_positions)
ax.set_xticklabels([average_by_time.index[i].strftime("%H:%M") for i in tick_positions])
ax.set_title("Average speed by time of day across all sensors")
ax.set_xlabel("Time of day")
ax.set_ylabel("Average speed")
plt.tight_layout()


## Optional metadata tables

These files are useful later when we connect sensors spatially or build graph-aware models.


In [ ]:
sensor_locations = pd.read_csv(sensor_locations_path)
distances = pd.read_csv(distances_path)

print(f"Sensor locations shape: {sensor_locations.shape}")
print(f"Distances shape: {distances.shape}")
display(sensor_locations.head())
display(distances.head())


## Initial recommendation

Based on the course scope and the structure of METR-LA, a strong first milestone is:
- Task: short-horizon traffic speed forecasting
- Target: each sensor's speed 15 to 60 minutes ahead
- Split: chronological train/validation/test split
- Baseline: persistence model (predict the next value from the current value)

After running the notebook, fill in the notes below.


In [ ]:
notes = {
    "target_variable": "traffic speed",
    "forecast_horizon": "TODO",
    "split_strategy": "chronological",
    "baseline_model": "persistence / last observed value",
    "observations": [
        "TODO: summarize shape and coverage",
        "TODO: summarize missing-data issues",
        "TODO: summarize daily traffic pattern",
    ],
}
notes
